# Buổi 27 — Lab

Chạy từng ô từ trên xuống. Mỗi bước ứng với một mục trong tài liệu (mục 5). Bạn sửa `bayes.py`; các ô tự dùng bản mới.
Các bước có lấy mẫu MCMC mất từ vài giây tới vài phút.

In [ ]:
# sửa tệp .py trong code/ thì các ô sau tự dùng bản mới, không cần khởi động lại
%load_ext autoreload
%autoreload 2

## Bước 1 — Dữ liệu và prior predictive (mục 4.1–4.2)

In [ ]:
%matplotlib inline
import warnings

import bayes as by
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
C = by.chia_car_parts(by.doc_car_parts())
thuc = np.concatenate(C["hoc"])
print("mã:", len(C["hoc"]), "| tháng học:", len(thuc), "| tỷ lệ tháng bằng 0:", round(float((thuc == 0).mean()), 3))
v = by.tien_nghiem(by.mo_hinh_phan_cap(C["hoc"]))
print("PRIOR =", by.PRIOR, "→ prior predictive: trung vị", np.quantile(v, 0.5), "| q90", f"{np.quantile(v, 0.9):.3g}",
      "| q99", f"{np.quantile(v, 0.99):.3g}", "| dữ liệu thật: lớn nhất", thuc.max())

## Bước 2 — Chẩn đoán hội tụ: ví dụ 8 trường (mục 4.3)

In [ ]:
for th in ("tam", "lech"):
    cd = by.chan_doan(by.lay_mau(by.mo_hinh_8_truong(th), target_accept=0.95), ["mu", "tau", "theta"])
    print(th, {k: round(v, 3) for k, v in cd.items()}, "→ dùng được:", by.dung_duoc(cd))
thu = {"r_hat_max": 1.002, "ess_bulk_min": 3000.0, "divergence": 27}
print("cổng chẩn đoán với", thu, "→ dùng được:", by.dung_duoc(thu))

## Bước 3 — Poisson phân cấp: không gộp, gộp hoàn toàn, gộp một phần (mục 4.4)

In [ ]:
_, idt = by.hoc_phan_cap(C["hoc"])
cd = by.chan_doan(idt, ["mu", "tau", "theta"])
print({k: round(v, 4) for k, v in cd.items()}, "→ dùng được:", by.dung_duoc(cd))
L = by.ba_cach_gop(C["hoc"], idt)
for k, v in L.items():
    print(f"{k:14s} RMSE {by.rmse(C['kiem'], v):.3f}", {n: round(by.rmse(C['kiem'], v, C['lich_su'] == n), 3) for n in by.LICH_SU})
print(by.tan_suat_dem(idt, C["kiem"]).round(3).to_string(index=False))

## Bước 4 — BSTS cho lượt thuê xe (mục 4.5)

Khoảng 3–5 phút.

In [ ]:
df = by.doc_xe_dap()
mo, ib, dub = by.bsts(df)
print({k: round(v, 4) for k, v in by.chan_doan(ib, ["sigma_level_trend", "sigma_MeasurementError", "initial_level_trend"]).items()})
fo = np.exp(dub["forecast_observed"].values[..., 0].reshape(-1, by.NGAY_KIEM))
du_bs = pd.DataFrame({"giua": np.median(fo, 0), "lo": np.quantile(fo, 0.05, 0), "hi": np.quantile(fo, 0.95, 0)})
print("BSTS", by.cham_ngay(df, du_bs), "| ETS", by.cham_ngay(df, by.ets(df)))

## Bước 5 — Gaussian process (mục 4.6)

Khoảng 2–5 phút. Sửa `THANH_PHAN_GP` rồi chạy lại ô này.

In [ ]:
m, ig, cd = by.hoc_gp(df)
print(by.THANH_PHAN_GP, {k: round(v, 4) for k, v in cd.items()}, "→ dùng được:", by.dung_duoc(cd))
du_gp = by.du_bao_gp(m, ig)
print("GP", by.cham_ngay(df, du_gp))
x = df["ds"].iloc[-by.NGAY_KIEM:]
plt.figure(figsize=(10, 3))
plt.plot(df["ds"].iloc[-120:], df["y"].iloc[-120:], color="black", lw=1)
plt.plot(x, du_gp["giua"])
plt.fill_between(x, du_gp["lo"], du_gp["hi"], alpha=0.2)
plt.show()

## Bước 6 — Kiểm tra

Trong terminal, thư mục `lab/`: `python lab.py check` — xanh 5/5 là xong (khoảng 3–5 phút).